# Prepare request text for embedding

**Question:** which NYC 311 fields become the text we embed, and what does indexing a day of heat requests cost?

**Data:** one complete calendar day of `HEAT/HOT WATER` requests (2024-01-27) from Socrata `erm2-nwe9` — every
ticket created that day, not a sample.

**Rules this notebook obeys:**
- Only fields known when the request arrives may be embedded. Anything written after the agency acts is
  payload, never query text.
- Nominal identifiers (`incident_zip`) are exact filters, not similarity input.
- No street address, latitude or longitude is fetched; nothing location-like leaves this machine.
- Imports happen once. No code cell sits without a heading above it.

In [1]:
import hashlib, json, time

import httpx
import pandas as pd
from google import genai
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, FieldCondition, Filter, MatchValue, PointStruct, VectorParams

from resolveiq.config import REPO_ROOT, get_settings

settings = get_settings()
url = f"{settings.nyc311_base_url}/{settings.nyc311_dataset_id}.json"
client = genai.Client(api_key=settings.gemini_api_key.get_secret_value())
qdrant = QdrantClient(url=settings.qdrant_url)
raw_dir = REPO_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

print(f"repo  {REPO_ROOT}")
print(f"embed {settings.gemini_embed_model} @ {settings.embedding_dimensions} dims   key …{settings.key_tail}")
print(f"qdrant {settings.qdrant_url}   collection {settings.qdrant_collection}")

repo  C:\Users\mejdi\Documents\resolveiq
embed gemini-embedding-001 @ 3072 dims   key …KMWA
qdrant http://localhost:6333   collection resolveiq-cases-collection-00


## What text the source actually gives us

The 311 open-data table has no free-text description field — all 48 columns are categorical or dates. The one
prose field, `resolution_description`, is only written after the agency acts, so it can never be part of the
query text. Before choosing what to embed, measure which intake fields exist and how much they vary.

In [2]:
INTAKE = ["complaint_type", "descriptor", "descriptor_2", "location_type", "address_type",
          "facility_type", "borough", "city", "incident_zip", "open_data_channel_type", "agency"]
POST = ["status", "closed_date", "resolution_description", "resolution_action_updated_date"]

PULL_PARAMS = {
    "$select": ", ".join(["unique_key", "created_date"] + INTAKE + POST),
    "$where": ("lower(complaint_type) = 'heat/hot water' "
               "and created_date >= '2024-01-27T00:00:00' "
               "and created_date < '2024-01-28T00:00:00'"),
    "$order": "unique_key",
    "$limit": "5000",
}

response = httpx.get(url, params=PULL_PARAMS, timeout=120.0)
if response.status_code >= 400:
    print(response.text[:500])
response.raise_for_status()

day = pd.DataFrame(response.json())
for col in INTAKE + POST:
    if col not in day:
        day[col] = pd.NA
print(f"{len(day):,} rows   {len(day.columns)} columns   one complete quiet day, not a sample")

798 rows   17 columns   one complete quiet day, not a sample


In [3]:
def profile(col):
    ser = day[col].fillna("").astype(str).str.strip()
    filled = ser != ""
    ln = ser[filled].str.len()
    return {"filled": f"{filled.mean():6.1%}", "distinct": ser[filled].nunique(),
            "chars_med": float(ln.median()) if len(ln) else 0,
            "top_share": f"{ser[filled].value_counts().iloc[0] / len(ser):6.1%}" if filled.any() else "—",
            "example": ser[filled].iloc[0][:44] if filled.any() else "—"}

audit = pd.DataFrame({c: profile(c) for c in INTAKE + POST}).T
print(audit.to_string())

                                filled distinct chars_med top_share                                       example
complaint_type                  100.0%        1      14.0    100.0%                                HEAT/HOT WATER
descriptor                      100.0%        2      15.0     69.0%                                APARTMENT ONLY
descriptor_2                    100.0%        3       7.0     52.4%                                       NO HEAT
location_type                   100.0%        1      20.0    100.0%                          RESIDENTIAL BUILDING
address_type                    100.0%        1       7.0    100.0%                                       ADDRESS
facility_type                     0.0%        0         0         —                                             —
borough                         100.0%        5       6.0     35.7%                                         BRONX
city                            100.0%       29       8.0     35.7%                     

## Choosing the text to embed

A field earns a place by changing between rows and by being known at intake. Measured on this day:

- **In:** `complaint_type`, `location_type`, `descriptor`, `descriptor_2`, `borough`. The first two are constant
  inside a heat-only index but vary across complaint types, so they keep their place for the mixed index later.
- **Out — `city`.** It equals `borough` on 63.7% of rows and holds 25 distinct neighbourhoods for Queens while
  the other four boroughs have one value each. One column, two granularities: Queens tickets would carry detail
  the rest cannot, and retrieval would look sharper there for a non-semantic reason.
- **Out — `incident_zip`.** The most discriminating field in the table (128 distinct, top share 4.3%) and the
  worst candidate for a similarity space: a vector would decide `10454` resembles `10453` for no reason.
  Nominal identifiers are exact filters, so it stays in the payload.
- **Out — `address_type`, `agency`, `facility_type`.** One distinct value each, or no values at all.
  `facility_type` is never returned: Socrata omits keys whose value is null, so a field empty across a slice
  becomes a missing *column*, not a NaN.
- **Out — everything post-closure.** `resolution_description` is 100% filled and the only prose, but it has
  **10 distinct values across 798 tickets** — a template, not a narrative. Payload only.

In [4]:
EMBED_FIELDS = ["complaint_type", "location_type", "descriptor", "descriptor_2", "borough"]
PAYLOAD_FIELDS = ["unique_key", "created_date", "borough", "city", "incident_zip", "address_type", "agency",
                  "open_data_channel_type", "status", "closed_date",
                  "resolution_description", "resolution_action_updated_date"]

def build_text(row):
    parts = [str(row[f]).strip() for f in EMBED_FIELDS if pd.notna(row[f]) and str(row[f]).strip()]
    return ", ".join(parts)

day["text_to_embed"] = day.apply(build_text, axis=1)
lens = day["text_to_embed"].str.len()
print(f"{len(day)} rows -> {day['text_to_embed'].nunique()} distinct texts")
print(f"length: median {lens.median():.0f}, min {lens.min()}, max {lens.max()}, empty {(lens == 0).sum()}")
print(day["text_to_embed"].value_counts().head(5).to_string())

798 rows -> 30 distinct texts
length: median 73, min 68, max 94, empty 0
text_to_embed
HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT AND NO HOT WATER, BRONX        94
HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT, BRONX                         91
HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT, BROOKLYN                      73
HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT, MANHATTAN                     67
HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT AND NO HOT WATER, MANHATTAN    55


In [5]:
sample = day["text_to_embed"].drop_duplicates().head(4).tolist()

try:
    started = time.perf_counter()
    batch = client.models.embed_content(model=settings.gemini_embed_model, contents=sample)
    batch_s = time.perf_counter() - started
    print(f"batch: {len(sample)} texts in {batch_s:.2f}s -> {len(batch.embeddings)} vectors, "
          f"dims {sorted({len(e.values) for e in batch.embeddings})}")
except Exception as exc:
    batch_s = None
    print(f"batch rejected: {type(exc).__name__}: {str(exc)[:220]}")

started = time.perf_counter()
for text in sample:
    client.models.embed_content(model=settings.gemini_embed_model, contents=text)
loop_s = time.perf_counter() - started

n = day["text_to_embed"].nunique()
print(f"loop : {len(sample)} texts in {loop_s:.2f}s")
print(f"all {n} distinct texts: "
      + (f"~{batch_s / len(sample) * n:.0f}s batched or " if batch_s else "")
      + f"~{loop_s / len(sample) * n:.0f}s one at a time")

batch: 4 texts in 1.43s -> 4 vectors, dims [3072]
loop : 4 texts in 2.06s
all 30 distinct texts: ~11s batched or ~15s one at a time


## What is a row in the index

30 distinct intake texts describe 798 tickets, and this category has only 10 resolution templates. So embedding
cost is not the constraint — whichever unit we choose, we embed the distinct texts once and reuse those vectors.
The real choice is what a retrieved hit should *be*: a past ticket, a distinct request text, or a
request-to-outcome pattern.

In [6]:
units = pd.DataFrame({
    "tickets": [len(day)],
    "distinct request texts": [day["text_to_embed"].nunique()],
    "distinct resolution templates": [day["resolution_description"].nunique()],
    "distinct (text, resolution) pairs": [day.groupby(["text_to_embed", "resolution_description"]).ngroups],
})
print(units.T.to_string())

per_text = day.groupby("text_to_embed")["resolution_description"].nunique()
pairs = day.groupby(["text_to_embed", "resolution_description"]).size()
print(f"\nresolutions per text   median {per_text.median():.0f}   max {per_text.max()}")
print(f"tickets per pair       median {pairs.median():.0f}   max {pairs.max()}")

                                     0
tickets                            798
distinct request texts              30
distinct resolution templates       10
distinct (text, resolution) pairs  161

resolutions per text   median 6   max 9
tickets per pair       median 2   max 42


## Index table: one row per ticket

Unit chosen: **ticket**. A retrieved hit should be a past request with its real outcome attached, not a category
label. The 798 tickets come from 30 distinct intake texts, so each text is embedded once and its vector reused —
798 rows for 30 calls' worth of work. Stated up front so it cannot be forgotten later: the index can hold every
ticket, but similarity can only ever distinguish 30 request shapes.

In [7]:
def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / ((sum(x * x for x in a) ** 0.5) * (sum(y * y for y in b) ** 0.5))

distinct_texts = day["text_to_embed"].drop_duplicates().tolist()

started = time.perf_counter()
batch = client.models.embed_content(model=settings.gemini_embed_model, contents=distinct_texts)
embed_s = time.perf_counter() - started

vectors = {t: e.values for t, e in zip(distinct_texts, batch.embeddings)}
assert len(vectors) == len(distinct_texts), "one vector per text expected"
assert {len(v) for v in vectors.values()} == {settings.embedding_dimensions}, "dims differ from .env"
print(f"{len(vectors)} distinct texts embedded in {embed_s:.1f}s -> {len(vectors) * settings.embedding_dimensions:,} floats")

# .embeddings is documented to follow input order; prove it instead of trusting it.
probe = distinct_texts[7]
fresh = client.models.embed_content(model=settings.gemini_embed_model, contents=probe).embeddings[0].values
scores = sorted(((cosine(fresh, vectors[t]), t) for t in distinct_texts), reverse=True)
print(f"order check: re-embedding one text matches its cached vector at {scores[0][0]:.4f}, "
      f"next best {scores[1][0]:.4f}")

30 distinct texts embedded in 2.4s -> 92,160 floats
order check: re-embedding one text matches its cached vector at 1.0000, next best 0.9821


In [8]:
index_df = day[PAYLOAD_FIELDS + ["text_to_embed"]].copy()
index_df["point_id"] = pd.to_numeric(index_df["unique_key"], errors="raise").astype("int64")
index_df["vector"] = index_df["text_to_embed"].map(vectors)

print(f"{len(index_df):,} tickets   point_id unique: {index_df['point_id'].is_unique}   "
      f"rows without a vector: {index_df['vector'].isna().sum()}")
print(f"columns: {list(index_df.columns)}")
index_df.drop(columns=["vector", "resolution_description"]).head(3).to_string()

798 tickets   point_id unique: True   rows without a vector: 0
columns: ['unique_key', 'created_date', 'borough', 'city', 'incident_zip', 'address_type', 'agency', 'open_data_channel_type', 'status', 'closed_date', 'resolution_description', 'resolution_action_updated_date', 'text_to_embed', 'point_id', 'vector']


'  unique_key             created_date   borough      city incident_zip address_type agency open_data_channel_type  status              closed_date resolution_action_updated_date                                                            text_to_embed  point_id\n0   60153653  2024-01-27T11:05:42.000     BRONX     BRONX        10454      ADDRESS    HPD                 ONLINE  Closed  2024-01-27T15:51:37.000        2024-01-27T00:00:00.000     HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT, BRONX  60153653\n1   60153654  2024-01-27T03:44:48.000  BROOKLYN  BROOKLYN        11210      ADDRESS    HPD                  PHONE  Closed  2024-01-28T17:48:13.000        2024-01-28T00:00:00.000  HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT, BROOKLYN  60153654\n2   60153655  2024-01-27T17:57:24.000    QUEENS   ASTORIA        11106      ADDRESS    HPD                 MOBILE  Closed  2024-01-30T14:01:22.000        2024-01-30T00:00:00.000    HEAT/HOT WATER, RESIDENTIAL BUIL

## Creating the collection

Cosine distance because we compare what a request means, not how long its text is. The size comes from
`settings.embedding_dimensions`, so a collection that disagrees with `.env` fails at creation rather than at
the first query. The `-00` suffix is the tutor's versioning habit: re-indexing bumps `00 → 01` rather than
silently overwriting a collection an older notebook may still be reading.

In [9]:
REBUILD = False   # True deletes the collection; § 12 is what puts the 798 points back

name = settings.qdrant_collection
if qdrant.collection_exists(name):
    if REBUILD:
        qdrant.delete_collection(name)
        print(f"deleted {name}")
    else:
        print(f"{name} already holds {qdrant.count(name).count:,} points — leaving it alone")
else:
    qdrant.create_collection(
        collection_name=name,
        vectors_config=VectorParams(size=settings.embedding_dimensions, distance=Distance.COSINE),
    )
    print(f"created {name}")

info = qdrant.get_collection(name)
print(f"{name}   status {info.status}   points {info.points_count:,}   segments {info.segments_count}")
print(f"  vectors {info.config.params.vectors}")

resolveiq-cases-collection-00 already holds 798 points — leaving it alone
resolveiq-cases-collection-00   status green   points 798   segments 8
  vectors size=3072 distance=<Distance.COSINE: 'Cosine'> hnsw_config=None quantization_config=None on_disk=None memory=None datatype=None multivector_config=None


## Upserting 798 tickets

Batches of 100 with `wait=True`, so the count read back in the next cell is the server's state, not a queue
estimate. The 311 `unique_key` becomes the Qdrant id, which makes a second run of this cell an update of the
same 798 points instead of a duplicated index. The payload carries the intake text plus `PAYLOAD_FIELDS`:
`text_to_embed` is stored so § 13 can group repeated shapes without a second lookup, and `unique_key` is
dropped because the id already holds it. No latitude, longitude or street name exists in the payload because
none was ever fetched.

In [10]:
BATCH = 100
name = settings.qdrant_collection

def clean(v):
    return None if pd.isna(v) else str(v)

payload_cols = ["text_to_embed"] + [c for c in PAYLOAD_FIELDS if c != "unique_key"]
points = [
    PointStruct(id=int(r.point_id), vector=r.vector,
                payload={c: clean(getattr(r, c)) for c in payload_cols})
    for r in index_df.itertuples(index=False)
]
print(f"{len(points):,} points built   payload keys: {payload_cols}")

started = time.perf_counter()
for n, i in enumerate(range(0, len(points), BATCH), start=1):
    qdrant.upsert(collection_name=name, points=points[i:i + BATCH], wait=True)
    print(f"  batch {n}: through {min(i + BATCH, len(points)):,}/{len(points):,}   "
          f"{time.perf_counter() - started:.1f}s")
print(f"upserted {len(points):,} points in {time.perf_counter() - started:.1f}s")

798 points built   payload keys: ['text_to_embed', 'created_date', 'borough', 'city', 'incident_zip', 'address_type', 'agency', 'open_data_channel_type', 'status', 'closed_date', 'resolution_description', 'resolution_action_updated_date']
  batch 1: through 100/798   0.2s
  batch 2: through 200/798   0.4s
  batch 3: through 300/798   0.7s
  batch 4: through 400/798   0.9s
  batch 5: through 500/798   1.1s
  batch 6: through 600/798   1.3s
  batch 7: through 700/798   1.5s
  batch 8: through 798/798   1.7s
upserted 798 points in 1.7s


In [11]:
count = qdrant.count(collection_name=name, exact=True).count
print(f"server holds {count:,} points   index table {len(index_df):,}   "
      + ("MATCH" if count == len(index_df) else "MISMATCH — stop here"))

row = index_df.iloc[0]
got = qdrant.retrieve(collection_name=name, ids=[int(row["point_id"])], with_payload=True, with_vectors=True)[0]
print(f"ticket {got.id} read back: vector {len(got.vector)} dims   payload {len(got.payload)} fields")
print("payload identical to the table:", all(got.payload[c] == clean(row[c]) for c in payload_cols))
print(f"vector vs local cache: cosine {cosine(got.vector, row['vector']):.8f}")
print("stored text:", got.payload["text_to_embed"])

open_tickets = index_df[index_df["closed_date"].isna()]
if len(open_tickets):
    probe = qdrant.retrieve(collection_name=name, ids=[int(open_tickets["point_id"].iloc[0])],
                           with_payload=True)[0]
    print(f"open ticket probe: closed_date -> {probe.payload['closed_date']!r}  "
          f"({len(open_tickets)} rows had pd.NA)")

server holds 798 points   index table 798   MATCH
ticket 60153653 read back: vector 3072 dims   payload 12 fields
payload identical to the table: True
vector vs local cache: cosine 1.00000000
stored text: HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT, BRONX
open ticket probe: closed_date -> None  (2 rows had pd.NA)


## Retrieving comparable tickets

A question in, ranked past tickets out. Two things have to be true for the result to mean anything. The query
must reach the same space the tickets were written in — but a manager types prose, while this index was built
from intake category strings, so the last cell tests that gap instead of assuming it. And scores can only be
read as ranks: the cache already showed two *different* categories at cosine 0.98, which the first cell turns
into a number.

In [12]:
pairs = [cosine(vectors[a], vectors[b])
         for i, a in enumerate(distinct_texts) for b in distinct_texts[i + 1:]]
band = pd.Series(pairs)
print(f"{len(band):,} pairs of DISTINCT texts   min {band.min():.4f}   median {band.median():.4f}   max {band.max():.4f}")
print(f"different categories never fall below {band.min():.3f}, so a hit at 0.98 is not 'similar' and a hit at "
      f"0.97 is not 'weak' — only the order carries information")

435 pairs of DISTINCT texts   min 0.8252   median 0.9055   max 0.9906
different categories never fall below 0.825, so a hit at 0.98 is not 'similar' and a hit at 0.97 is not 'weak' — only the order carries information


In [13]:
def retrieve_similar_cases(question, k=15, borough=None):
    """Free-text question in, ranked past tickets out. Written to move into apps/api unchanged."""
    qv = client.models.embed_content(
        model=settings.gemini_embed_model, contents=question
    ).embeddings[0].values
    qfilter = None
    if borough:
        qfilter = Filter(must=[FieldCondition(key="borough", match=MatchValue(value=borough.upper()))])
    res = qdrant.query_points(collection_name=settings.qdrant_collection, query=qv,
                              query_filter=qfilter, limit=k, with_payload=True)
    return pd.DataFrame([{"id": h.id, "score": h.score, **h.payload} for h in res.points])

In [14]:
question = "no heat in an entire apartment building in brooklyn"
hits = retrieve_similar_cases(question, k=15)
hits["created"] = pd.to_datetime(hits["created_date"])
hits["closed"] = pd.to_datetime(hits["closed_date"])
hits["hours"] = (hits["closed"] - hits["created"]).dt.total_seconds() / 3600

print(f"query: {question!r}\n")
print(hits[["score", "id", "borough", "text_to_embed", "status", "hours"]]
      .head(8).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

grouped = hits.groupby("text_to_embed").agg(hits=("score", "size"), best=("score", "max"),
                                            median_hours=("hours", "median"))
print(f"\n{len(hits)} tickets collapse to {len(grouped)} distinct request shapes:\n"
      + grouped.sort_values("best", ascending=False).to_string(float_format=lambda v: f"{v:.3f}"))

filtered = retrieve_similar_cases(question, k=15, borough="queens")
print(f"\nborough='queens' → {len(filtered)} hits, boroughs present {sorted(filtered['borough'].unique())} "
      f"(filtered inside the database, not sliced afterwards)")

as_categories = "HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT, BROOKLYN"
alt = retrieve_similar_cases(as_categories, k=15)
print(f"\nsame meaning, written the way the index was written: top-5 ids identical to the prose query: "
      f"{alt['id'].head(5).tolist() == hits['id'].head(5).tolist()}")

query: 'no heat in an entire apartment building in brooklyn'

 score       id  borough                                                            text_to_embed status   hours
0.9380 60153694 BROOKLYN HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT, BROOKLYN Closed 32.0483
0.9380 60153707 BROOKLYN HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT, BROOKLYN Closed 52.5514
0.9380 60155739 BROOKLYN HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT, BROOKLYN Closed  0.0000
0.9380 60155737 BROOKLYN HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT, BROOKLYN Closed 45.9694
0.9380 60155726 BROOKLYN HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT, BROOKLYN Closed 22.5522
0.9380 60153683 BROOKLYN HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT, BROOKLYN Closed 71.9442
0.9380 60158739 BROOKLYN HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT, BROOKLYN Closed 11.1475
0.9380 60158745 BROOKLYN HEAT/HOT 

In [15]:
probes = {
    "names brooklyn":   "no heat in an entire apartment building in brooklyn",
    "no borough named": "there is no heat at all in the building",
    "different words":  "the radiators stayed cold overnight in my apartment",
}
for label, q in probes.items():
    h = retrieve_similar_cases(q, k=25)
    g = h.groupby("text_to_embed")["score"].agg(["size", "max"]).sort_values("max", ascending=False)
    print(f"\n{label}: {q!r}")
    for text, r in g.iterrows():
        print(f"  {int(r['size']):>3}x  {r['max']:.4f}  {text}")


names brooklyn: 'no heat in an entire apartment building in brooklyn'
   25x  0.9380  HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT, BROOKLYN

no borough named: 'there is no heat at all in the building'
   25x  0.8043  HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT, MANHATTAN

different words: 'the radiators stayed cold overnight in my apartment'
   22x  0.7294  HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT, MANHATTAN
    3x  0.7280  HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT, QUEENS


In [16]:
def retrieve_case_shapes(question, k=8, group_size=3, borough=None):
    """Up to group_size past tickets for each of the k closest request shapes."""
    qv = client.models.embed_content(
        model=settings.gemini_embed_model, contents=question
    ).embeddings[0].values
    qfilter = None
    if borough:
        qfilter = Filter(must=[FieldCondition(key="borough", match=MatchValue(value=borough.upper()))])
    res = qdrant.query_points_groups(collection_name=settings.qdrant_collection,
                                     group_by="text_to_embed", query=qv, query_filter=qfilter,
                                     limit=k, group_size=group_size, with_payload=True)
    rows = [{"shape": g.id, "score": h.score, "ticket": h.id,
             "borough": h.payload["borough"], "status": h.payload["status"],
             "resolution": h.payload["resolution_description"][:58]}
            for g in res.groups for h in g.hits]
    return pd.DataFrame(rows)

q = "the radiators stayed cold overnight in my apartment"
print(f"{q!r}\n")
print(retrieve_case_shapes(q, k=5, group_size=2).to_string(index=False))

'the radiators stayed cold overnight in my apartment'

                                                                                    shape    score   ticket       borough status                                                 resolution
                 HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT, MANHATTAN 0.729387 60154629     MANHATTAN Closed The Department of Housing Preservation and Development was
                 HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT, MANHATTAN 0.729387 60155695     MANHATTAN Closed The complaint you filed is a duplicate of a condition alre
                    HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT, QUEENS 0.728009 60159746        QUEENS Closed The Department of Housing Preservation and Development con
                    HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT, QUEENS 0.728009 60158709        QUEENS Closed The Department of Housing Preservation and Development con
     

In [17]:
cold = "the place is freezing and nothing warmed up overnight"
overlap = sorted({w for w in cold.lower().split() if len(w) > 3} &
                 {w for t in distinct_texts for w in t.lower().replace(",", " ").split()})
print(f"query: {cold!r}\nshared words with the indexed text: {overlap or 'none'}\n")
out = retrieve_case_shapes(cold, k=6, group_size=1)
print(out[["score", "borough", "shape"]].to_string(index=False))

query: 'the place is freezing and nothing warmed up overnight'
shared words with the indexed text: none

   score   borough                                                                     shape
0.668144  BROOKLYN   HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT, BROOKLYN
0.667481    QUEENS     HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT, QUEENS
0.666177 MANHATTAN HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT, MANHATTAN
0.665822 MANHATTAN  HEAT/HOT WATER, RESIDENTIAL BUILDING, APARTMENT ONLY, NO HEAT, MANHATTAN
0.665404  BROOKLYN  HEAT/HOT WATER, RESIDENTIAL BUILDING, ENTIRE BUILDING, NO HEAT, BROOKLYN


## What this index can and cannot claim

The claims below are printed by the cell under this one, so they are re-derived on every run rather than typed
from memory.

In [18]:
band = pd.Series([cosine(vectors[a], vectors[b])
                  for i, a in enumerate(distinct_texts) for b in distinct_texts[i + 1:]])
probe_q = "the place is freezing and nothing warmed up overnight"
qv = client.models.embed_content(model=settings.gemini_embed_model, contents=probe_q).embeddings[0].values
prose_top = qdrant.query_points(collection_name=settings.qdrant_collection, query=qv, limit=1).points[0].score

print(f"{len(index_df):,} tickets / {len(distinct_texts)} distinct texts "
      f"= {len(index_df) / len(distinct_texts):.1f} tickets per distinguishable answer")
print(f"different-category pairs ({len(band):,}): min {band.min():.4f}  median {band.median():.4f}  max {band.max():.4f}")
print(f"correct prose answer above: {prose_top:.4f}  ->  below every pair of distinct categories: "
      f"{prose_top < band.min()}")
print(f"outcomes per request text: median {per_text.median():.0f} of {day['resolution_description'].nunique()} templates")

798 tickets / 30 distinct texts = 26.6 tickets per distinguishable answer
different-category pairs (435): min 0.8252  median 0.9055  max 0.9906
correct prose answer above: 0.6681  ->  below every pair of distinct categories: True
outcomes per request text: median 6 of 10 templates


- **30 answers, 798 tickets.** Every point is a real ticket, but similarity can only ever separate 30 request
  shapes. This is a shaped lookup over a category grid, not a search through free text.
- **Scores may not be thresholded and may not be shown as confidence.** A correct prose answer scored 0.668
  while two *different* categories score up to 0.98. Rank order is the only readable signal.
- **`borough` sits inside the embedded text and leaks into ranking.** A question naming no borough still produced
  `BROOKLYN 0.6681 > QUEENS 0.6675 > MANHATTAN 0.6662`. The `-01` rebuild tests removing it.
- **Intake text does not predict outcome.** Each request shape leads to a median of 6 of the 10 resolution
  templates. Retrieval may say *these past requests were written like yours, and here is what happened*; never
  *requests like yours get resolved like this*.
- **One quiet day, one complaint type.** 2024-01-27 held 798 of January's 51,325 heat requests, the lowest day
  of the month, so the 46.7 h vs 28.9 h event gap from notebook 1 is invisible here by construction.
- **What did hold up:** a query sharing no words at all with the index still returned heat requests at apartment
  scope, and grouping five shapes over 798 tickets surfaced four boroughs instead of one twenty-five times.
- **Closes are near-complete, so this is not right-censoring.** 99.7% of the day closed; the 2 open tickets
  stored `closed_date = None` rather than being dropped.

## Provenance

The pull and the collection each get a record on disk: what was asked, when, how many rows came back, and the
limitation of that particular file. A sidecar that repeats the exact params object the request used is evidence;
one with params retyped from memory is a guess that happens to be formatted nicely.

In [19]:
day_path = raw_dir / "heat_2024_01_27_day.json"
day.drop(columns=["text_to_embed"]).to_json(day_path, orient="records", indent=1)

sidecar = {
    "retrieved_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "dataset": settings.nyc311_dataset_id,
    "url": url,
    "params": PULL_PARAMS,
    "rows": len(day),
    "sha256": hashlib.sha256(day_path.read_bytes()).hexdigest(),
    "ordering": "$order=unique_key, single page; $limit 5000 exceeded the 798 rows so no paging",
    "limitation": "one calendar day, the lowest-volume heat day of Jan-2024; not a sample of the month and "
                  "no surge behaviour in it",
}
sidecar_path = day_path.with_name(day_path.stem + ".meta.json")
sidecar_path.write_text(json.dumps(sidecar, indent=2), encoding="utf-8")
print(day_path.name, f"{day_path.stat().st_size:,} bytes")
print(sidecar_path.name, "sha256", sidecar["sha256"][:16], "rows", sidecar["rows"])

heat_2024_01_27_day.json 662,653 bytes
heat_2024_01_27_day.meta.json sha256 fe7eef50029daa5b rows 798


In [20]:
idx_dir = REPO_ROOT / "data" / "indexes"
idx_dir.mkdir(parents=True, exist_ok=True)

info = qdrant.get_collection(settings.qdrant_collection)
index_meta = {
    "built_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "collection": settings.qdrant_collection,
    "server": settings.qdrant_url,
    "points": info.points_count,
    "vectors": str(info.config.params.vectors),
    "embedding_model": settings.gemini_embed_model,
    "embedded_fields": EMBED_FIELDS,
    "payload_fields": payload_cols,
    "distinct_texts": len(distinct_texts),
    "built_from": day_path.name,
    "known_limits": [f"{len(distinct_texts)} distinct texts across {len(index_df)} tickets",
                     "borough embedded in the text, so ranking leaks borough order; -01 tests removing it",
                     "intake text does not predict resolution template"],
}
meta_path = idx_dir / f"{settings.qdrant_collection}.meta.json"
meta_path.write_text(json.dumps(index_meta, indent=2), encoding="utf-8")
print(meta_path)
print(json.dumps(index_meta, indent=2))

C:\Users\mejdi\Documents\resolveiq\data\indexes\resolveiq-cases-collection-00.meta.json
{
  "built_at": "2026-09-23T11:58:46.497819+00:00",
  "collection": "resolveiq-cases-collection-00",
  "server": "http://localhost:6333",
  "points": 798,
  "vectors": "size=3072 distance=<Distance.COSINE: 'Cosine'> hnsw_config=None quantization_config=None on_disk=None memory=None datatype=None multivector_config=None",
  "embedding_model": "gemini-embedding-001",
  "embedded_fields": [
    "complaint_type",
    "location_type",
    "descriptor",
    "descriptor_2",
    "borough"
  ],
  "payload_fields": [
    "text_to_embed",
    "created_date",
    "borough",
    "city",
    "incident_zip",
    "address_type",
    "agency",
    "open_data_channel_type",
    "status",
    "closed_date",
    "resolution_description",
    "resolution_action_updated_date"
  ],
  "distinct_texts": 30,
  "built_from": "heat_2024_01_27_day.json",
  "known_limits": [
    "30 distinct texts across 798 tickets",
    "borou